## Step1-Process-RACMO

Takes RACMO surface runoff and sums over ISMIP7 drainage basins to generate monthly 1958-2014 per-basin subglacial discharge from RACMO, which is used for bias-correction later in the processing. There is code to make a number of plots along the way that may be useful in sense-checking or debugging; this code is commented by default. Script takes around 10 minutes to run on a laptop, most of this is the last code cell.

Creator: Donald Slater, donald.slater@ed.ac.uk, 8 July 2025.

Cleaned up by Donald Slater, 9 Feb 2026.

#### Required files/inputs
- RACMO surface runoff files, assumed to be of form _runoff.????.*.nc_
- Drainage basins definition file *ismip7_runoff_basins_nearestbelowsl.nc* (assumed in same directory as this script)

#### Outputs
- RACMO monthly 1958-2014 per-basin subglacial discharge *Q_RACMO_1958_2014.nc*.

#### Setup

In [1]:
# directory that contains the original RACMO surface runoff data
racmo_runoff_dir = '/Users/dslater2/Documents/RACMO/'

# directory for output file
output_runoff_dir = '/Users/dslater2/Documents/RACMO/'


#### Imports

In [2]:
# libraries
import netCDF4 as nc
from netCDF4 import Dataset
import numpy as np
import matplotlib.pyplot as plt
import os
import glob
from scipy.interpolate import griddata

#### Get ISMIP7 drainage basins that have been pre-generated in matlab scripts

In [3]:
# load drainage basins
basins_file = 'ismip7_runoff_basins_nearestbelowsl.nc'
basins = Dataset(basins_file, mode='r')
x = basins.variables['x'][:].filled(np.nan)
y = basins.variables['y'][:].filled(np.nan)
b = basins.variables['basin'][:].filled(np.nan)
m = basins.variables['icesheetmask'][:].filled(np.nan)
o = basins.variables['convexhullmask'][:].filled(np.nan)

# get spatial indices corresponding to each basin (used later)
b_unique = np.unique(b[:])
basin_indices = {}
for i in b_unique:
    basin_indices[i] = np.where(b==i)

# indices outside convex hull
outside_indices = np.where(o==1)

# plot drainage basins and mask
""" plt.figure(figsize=(10, 5))
plt.subplot(131)
plt.pcolormesh(x, y, m, cmap='gray')
plt.axis('equal')
plt.title('Ice Sheet Mask')
plt.colorbar(label='mask value')
plt.subplot(132)
plt.pcolormesh(x, y, o, cmap='gray')
plt.axis('equal')
plt.title('Convex Hull Mask')
plt.colorbar(label='mask value')
plt.subplot(133)
plt.title('Drainage Basins')
plt.pcolormesh(x, y, b)
plt.axis('equal')
plt.colorbar(label='basin id')
plt.tight_layout()
plt.show() """

" plt.figure(figsize=(10, 5))\nplt.subplot(131)\nplt.pcolormesh(x, y, m, cmap='gray')\nplt.axis('equal')\nplt.title('Ice Sheet Mask')\nplt.colorbar(label='mask value')\nplt.subplot(132)\nplt.pcolormesh(x, y, o, cmap='gray')\nplt.axis('equal')\nplt.title('Convex Hull Mask')\nplt.colorbar(label='mask value')\nplt.subplot(133)\nplt.title('Drainage Basins')\nplt.pcolormesh(x, y, b)\nplt.axis('equal')\nplt.colorbar(label='basin id')\nplt.tight_layout()\nplt.show() "

#### Read RACMO grid and get basins on RACMO grid

RACMO surface runoff has same projection but is on different grid. Read the RACMO grid and interpolate the ISMIP7 basins onto the RACMO grid.

In [4]:
# directory and files
runoff_files = sorted(glob.glob(os.path.join(racmo_runoff_dir, 'runoff.????.*.nc')))

# load RACMO runoff grid
racmo = Dataset(runoff_files[0], mode='r')
x_racmo = racmo.variables['lon'][:].filled(np.nan) # note that this is indeed x (in m) despite it being called lon in the file
y_racmo = racmo.variables['lat'][:].filled(np.nan)

# interpolate basins and masks (which are on ISMIP grid) onto RACMO grid
[X, Y] = np.meshgrid(x, y)
[X_racmo, Y_racmo] = np.meshgrid(x_racmo, y_racmo)
b_racmo = griddata((X.flatten(), Y.flatten()), b.flatten(), (X_racmo, Y_racmo), method='nearest')
m_racmo = griddata((X.flatten(), Y.flatten()), m.flatten(), (X_racmo, Y_racmo), method='nearest')
o_racmo = griddata((X.flatten(), Y.flatten()), o.flatten(), (X_racmo, Y_racmo), method='nearest')

# get spatial indices corresponding to each basin and within main ice sheet (avoids including runoff from ice caps)
b_unique_racmo = np.unique(b_racmo[:])
basin_indices_racmo = {}
for i in b_unique_racmo:
    basin_indices_racmo[i] = np.where((b_racmo==i) & (m_racmo==1))

# plot drainage basins and mask on racmo grid to check
""" plt.figure(figsize=(10, 5))
plt.subplot(131)
plt.pcolormesh(x_racmo, y_racmo, m_racmo, cmap='gray')
plt.axis('equal')
plt.title('Ice Sheet Mask')
plt.colorbar(label='mask value')
plt.subplot(132)
plt.pcolormesh(x_racmo, y_racmo, o_racmo, cmap='gray')
plt.axis('equal')
plt.title('Convex Hull Mask')
plt.colorbar(label='mask value')
plt.subplot(133)
plt.title('Drainage Basins')
plt.pcolormesh(x_racmo, y_racmo, b_racmo)
plt.axis('equal')
plt.colorbar(label='basin id')
plt.tight_layout()
plt.show() """

" plt.figure(figsize=(10, 5))\nplt.subplot(131)\nplt.pcolormesh(x_racmo, y_racmo, m_racmo, cmap='gray')\nplt.axis('equal')\nplt.title('Ice Sheet Mask')\nplt.colorbar(label='mask value')\nplt.subplot(132)\nplt.pcolormesh(x_racmo, y_racmo, o_racmo, cmap='gray')\nplt.axis('equal')\nplt.title('Convex Hull Mask')\nplt.colorbar(label='mask value')\nplt.subplot(133)\nplt.title('Drainage Basins')\nplt.pcolormesh(x_racmo, y_racmo, b_racmo)\nplt.axis('equal')\nplt.colorbar(label='basin id')\nplt.tight_layout()\nplt.show() "

#### Read runoff and estimate subglacial discharge for single file to check process

Now we can estimate the subglacial discharge for an ISMIP7 basin by summing the RACMO surface runoff over the basins indices based on the RACMO grid. Do this for a single file to check the processing. Could skip to next code block if just running the processing.

In [5]:
# load runoff
runoff = Dataset(runoff_files[0], mode='r')
xr = runoff.variables['lon'][:].filled(np.nan)
yr = runoff.variables['lat'][:].filled(np.nan)
t = runoff.variables['time'][:].filled(np.nan)
r = runoff.variables['runoffcorr'][:].filled(np.nan)

# check coordinate system for runoff is the same as for the basins
if (np.sum(xr-x_racmo)!=0) | (np.sum(yr-y_racmo)!=0):
    print('Warning: runoff coordinates not as expected')
    
# convert runoff units from mmwe/month to m3/s (assumes 365/12 days in a month)
r = 1000*1000*r/(1000*(365/12)*86400)

# sum racmo runoff values over drainage basins on racmo grid, and assign values to drainage basins on ismip grid
Q = np.zeros((r.shape[0], len(y), len(x))) # initialize subglacial discharge array on ismip grid
for i in b_unique: # loop over drainage basins
    inds = basin_indices[i] # basin indices on ismip grid
    inds_racmo = basin_indices_racmo[i] # basin indices on racmo grid
    for k in range(r.shape[0]): # loop over time stamps
        Q[k,inds[0],inds[1]] = np.sum(r[k,inds_racmo[0],inds_racmo[1]]) # assign subglacial discharge

# assign outside of convex hull values to nominal value (which is total runoff split over 50 basins)
for k in range(r.shape[0]):
    Q[k,outside_indices[0],outside_indices[1]] = np.sum(np.unique(Q[k,:,:]))/50

# plot an example subglacial discharge field (typical values 100s-1000s m3/s in mid-summer)
""" plt.pcolormesh(x,y,Q[8,:,:])
plt.colorbar(label='subglacial discharge (m$^3$/s)')
plt.axis('equal')
plt.show() """

" plt.pcolormesh(x,y,Q[8,:,:])\nplt.colorbar(label='subglacial discharge (m$^3$/s)')\nplt.axis('equal')\nplt.show() "

#### Now process all RACMO files and save out

Note that on a laptop this takes about 10 minutes for 60 years of data and the output file is about 180 MB.

In [6]:
# output file
output_runoff_file = output_runoff_dir + 'Q_RACMO_1958_2014.nc'

# delete existing output file if it exists
if os.path.exists(output_runoff_file):
    os.remove(output_runoff_file)

# create output netcdf
with Dataset(output_runoff_file,'w',format='NETCDF4') as nc_out:
    
    # dimensions
    nc_out.createDimension('x', len(x))
    nc_out.createDimension('y', len(y))
    nc_out.createDimension('time', None) # unlimited time dimension

    # variables
    x_var = nc_out.createVariable('x', np.int32, ('x',)) # int32 sufficient for coords
    y_var = nc_out.createVariable('y', np.int32, ('y',))
    time_var = nc_out.createVariable('time', np.float32, ('time',))
    Q_var = nc_out.createVariable('Q', np.float32, ('time', 'y', 'x'),zlib=True, complevel=9)

    # metadata
    x_var.standard_name = 'projection_x_coordinate'
    x_var.units = 'meters'
    x_var.long_name = 'x coordinate in EPSG:3413 projection'
    x_var.axis = 'X'

    y_var.standard_name = 'projection_y_coordinate'
    y_var.units = 'meters'
    y_var.long_name = 'y coordinate in EPSG:3413 projection'
    y_var.axis = 'Y'

    time_var.standard_name = 'time'
    time_var.units = 'months since 1850-01-15 00:00:00'
    time_var.calendar = 'noleap'
    time_var.long_name = 'time'

    Q_var.standard_name = 'subglacial_discharge_volume_flux'
    Q_var.units = 'm3 s-1'
    Q_var.long_name = 'basin subglacial discharge'
    Q_var.grid_mapping = 'crs'

    # write coordinates
    x_var[:] = np.array(x, dtype=np.int32)
    y_var[:] = np.array(y, dtype=np.int32)

    # Create grid mapping variable with EPSG:3413 info
    crs_var = nc_out.createVariable('crs', 'i4')
    crs_var.grid_mapping_name = 'polar_stereographic'
    crs_var.straight_vertical_longitude_from_pole = -45.0
    crs_var.latitude_of_projection_origin = 90.0
    crs_var.standard_parallel = 70.0
    crs_var.false_easting = 0.0
    crs_var.false_northing = 0.0
    crs_var.semi_major_axis = 6378137.0
    crs_var.inverse_flattening = 298.257223563

    # Global attributes
    nc_out.title = 'Monthly Basin Subglacial Discharge from RACMO (1958-2014)'
    nc_out.history = 'Created ' + str(np.datetime64('now'))

# Initialize time index for appending
time_index = 0

# loop over runoff files
for file_path in runoff_files:
    print(f"Processing {file_path}")

    # extract year from file name
    year = int(os.path.basename(file_path).split('.')[1])

    with Dataset(file_path, 'r') as runoff:
        
        # read data for this year noting that the name of the coordinate varies by year
        if year < 1990:
            xr = runoff.variables['lon'][:].filled(np.nan)
            yr = runoff.variables['lat'][:].filled(np.nan)
        else:
            xr = runoff.variables['x'][:].filled(np.nan)
            yr = runoff.variables['y'][:].filled(np.nan)
        # time and runoff are named consistently
        t = runoff.variables['time'][:].filled(np.nan)
        r = runoff.variables['runoffcorr'][:].filled(np.nan)

        # convert runoff units mmwe/month to m3/s
        r = 1000 * 1000 * r / (1000 * (365 / 12) * 86400)

        # convert time to months since 1850-01-15
        if year < 1990: # pre-1990 the format is just month of year
            t = t + 12*(year-1850)
        else: # post-1990 the format is just doy
            t = 12*(year-1850) + np.round((t-15)/30)

        # initialize output Q array for this file
        Q = np.zeros((r.shape[0], len(y), len(x)))

        # sum runoff over basins
        for i in b_unique: # loop over drainage basins
            inds = basin_indices[i] # basin indices on ismip grid
            inds_racmo = basin_indices_racmo[i] # basin indices on racmo grid
            for k in range(r.shape[0]): # loop over time stamps
                Q[k,inds[0],inds[1]] = np.sum(r[k,inds_racmo[0],inds_racmo[1]]) # assign subglacial discharge

        # assign outside of convex hull values to nominal value (which is total runoff split over 50 basins)
        for k in range(r.shape[0]):
            Q[k,outside_indices[0],outside_indices[1]] = np.sum(np.unique(Q[k,:,:]))/50

    # append data to output netcdf
    with Dataset(output_runoff_file, 'a') as nc_out:
        nc_out.variables['time'][time_index:time_index + len(t)] = t.astype(np.float32)
        nc_out.variables['Q'][time_index:time_index + len(t), :, :] = Q.astype(np.float32)

    time_index += len(t)

print("All files processed and appended.")

Processing /Users/dslater2/Documents/RACMO/runoff.1958.BN_RACMO2.3p2_FGRN055_1km.MM.nc
Processing /Users/dslater2/Documents/RACMO/runoff.1959.BN_RACMO2.3p2_FGRN055_1km.MM.nc
Processing /Users/dslater2/Documents/RACMO/runoff.1960.BN_RACMO2.3p2_FGRN055_1km.MM.nc
Processing /Users/dslater2/Documents/RACMO/runoff.1961.BN_RACMO2.3p2_FGRN055_1km.MM.nc
Processing /Users/dslater2/Documents/RACMO/runoff.1962.BN_RACMO2.3p2_FGRN055_1km.MM.nc
Processing /Users/dslater2/Documents/RACMO/runoff.1963.BN_RACMO2.3p2_FGRN055_1km.MM.nc
Processing /Users/dslater2/Documents/RACMO/runoff.1964.BN_RACMO2.3p2_FGRN055_1km.MM.nc
Processing /Users/dslater2/Documents/RACMO/runoff.1965.BN_RACMO2.3p2_FGRN055_1km.MM.nc
Processing /Users/dslater2/Documents/RACMO/runoff.1966.BN_RACMO2.3p2_FGRN055_1km.MM.nc
Processing /Users/dslater2/Documents/RACMO/runoff.1967.BN_RACMO2.3p2_FGRN055_1km.MM.nc
Processing /Users/dslater2/Documents/RACMO/runoff.1968.BN_RACMO2.3p2_FGRN055_1km.MM.nc
Processing /Users/dslater2/Documents/RACMO/